In [1]:
from mc_experiment import (
    make_seed_counter,
    next_seed,
    standardize_innovations,
    summarize_reference_experiment,
    summarize_mle_augmentation_experiment,
    augmented_config_path,
)

from SymbolicDSGE import ModelParser, DSGESolver, Shock
from SymbolicDSGE.bayesian import make_prior

from numpy import log
import numpy as np

from scipy.stats import chi2, gaussian_kde, norm

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
import cProfile

import contextlib
import io
STDOUT_VOID = lambda: contextlib.redirect_stdout(io.StringIO())

_KNOWN_R = False
_AUGMENTED_PARAM = 'r_coef'
_AUGMENTED_EQUATION = 'Rate'
_AUGMENTED_CONFIG = augmented_config_path(_AUGMENTED_EQUATION)
_MEAS_ERR_SCALE = 0.25
_MC_SAMPLES = 100_000
_MC_ALPHA = 0.05
_MC_SUMMARY_ONLY = True
_MC_INCLUDE_BY_PREDICTOR = False
_FIGSIZE_1D = (10, 6)
_FIGSIZE_2D = (12, 6)


Detected IPython. Loading juliacall extension. See https://juliapy.github.io/PythonCall.jl/stable/compat/#IPython


In [2]:
# Load reference model
parser = ModelParser("../../MODELS/misspec_test/reference.yaml")
config, kalman = parser.get_all()
solver = DSGESolver(config, kalman)

comp = solver.compile(
    n_state=3,
    n_exog=3,
)
sol = solver.solve(
    comp,
    steady_state=[0.0, 0.0, 0.0, 0.0, 0.0],
)

print("Transition matrix:\n", sol.A.round(3), "\n")
print("Shock Loadings:\n", sol.B.round(3))

Transition matrix:
 [[ 0.83  -0.     0.     0.     0.   ]
 [ 0.     0.85   0.     0.     0.   ]
 [ 0.288 -0.047  0.28   0.     0.   ]
 [ 0.892  0.708 -1.711  0.     0.   ]
 [ 0.7   -0.115 -1.363  0.     0.   ]] 

Shock Loadings:
 [[ 1.     0.     0.   ]
 [ 0.     1.     0.   ]
 [ 0.     0.     1.   ]
 [ 3.193  0.493 -6.107]
 [ 2.531 -0.406 -4.864]]


In [3]:
# Load Misspecified DGP
parser_dgp = ModelParser("../../MODELS/misspec_test/misspec.yaml")
config_dgp, kalman_dgp = parser_dgp.get_all()
solver_dgp = DSGESolver(config_dgp, kalman_dgp)
comp_dgp = solver_dgp.compile(
    n_state=3,
    n_exog=3,
)
sol_dgp = solver_dgp.solve(
    comp_dgp,
    steady_state=[0.0, 0.0, 0.0, 0.0, 0.0],
)

In [4]:
# Large sample simulations used to approximate measurement-noise variances
_large_sample_seed_counter = make_seed_counter(start=100_000)
shocks_large = {
    "g,z": Shock(10_000, "norm", multivar=True, seed=next_seed(_large_sample_seed_counter)).shock_generator(),
    "r": Shock(10_000, "norm", multivar=False, seed=next_seed(_large_sample_seed_counter)).shock_generator(),
}

sim1 = sol_dgp.sim(
    T=10_000,
    shocks=shocks_large,
    observables=True,
)

sim2 = sol.sim(
    T=10_000,
    shocks=shocks_large,
    observables=True,
)

In [5]:
T = 200
_plot_seed_counter = make_seed_counter(start=2_000_000)

err_var = np.var(np.column_stack([sim1["OutGap"], sim1["Infl"], sim1["Rate"]]), axis=0)
mc_reference = summarize_reference_experiment(
    sol,
    sol_dgp,
    T=T,
    err_var=err_var,
    meas_err_scale=_MEAS_ERR_SCALE,
    mc_samples=_MC_SAMPLES,
    known_r=_KNOWN_R,
    alpha=_MC_ALPHA,
    summary_only=_MC_SUMMARY_ONLY,
    include_by_predictor=_MC_INCLUDE_BY_PREDICTOR,
)

rep_ref = mc_reference["representative"]
sim_dgp = rep_ref.sim_dgp
obs = rep_ref.obs
kf = rep_ref.kf
std_innov = rep_ref.std_innov
err_scale = rep_ref.err_scale
N, n_obs = kf.innov.shape

_measurement_order = {"OutGap": 0, "Infl": 1, "Rate": 2}
_predictor_order = {"Pi": 0, "x": 1, "r": 2}

def _sort_summary(df):
    out = df.copy()
    if "measurement" in out.columns:
        out["measurement_order"] = out["measurement"].map(_measurement_order)
    if "predictor" in out.columns:
        out["predictor_order"] = out["predictor"].map(_predictor_order)
    if "target" in out.columns:
        out["target_order"] = out["target"].map(_predictor_order)
    if "regressor" in out.columns:
        out["regressor_order"] = out["regressor"].map(_predictor_order)
    sort_cols = [
        col
        for col in ["measurement_order", "target_order", "predictor_order", "regressor_order"]
        if col in out.columns
    ]
    if sort_cols:
        out = out.sort_values(sort_cols)
    return out.drop(columns=[col for col in ["measurement_order", "target_order", "predictor_order", "regressor_order"] if col in out.columns])

sim_ref = sol.sim(
    T=T,
    shocks={
        "g,z": Shock(T, "norm", multivar=True, seed=next_seed(_plot_seed_counter)).shock_generator(),
        "r": Shock(T, "norm", multivar=False, seed=next_seed(_plot_seed_counter)).shock_generator(),
    },
    observables=True,
)
ref = np.column_stack([sim_ref["OutGap"], sim_ref["Infl"], sim_ref["Rate"]])[1:, :]

obs_dgp = np.column_stack([sim1["OutGap"], sim1["Infl"], sim1["Rate"]])[1:, :]
if np.any(err_scale != 0.0):
    _plot_rng = np.random.default_rng(next_seed(_plot_seed_counter))
    obs_dgp = obs_dgp + _plot_rng.normal(scale=np.sqrt(err_scale), size=obs_dgp.shape)

In [6]:
print(f"Known R assumption: {_KNOWN_R}")
print(f"Augmented measurement equation: {_AUGMENTED_EQUATION}")
print(f"Augmented coefficient: {_AUGMENTED_PARAM}")
print(f"Monte Carlo replications: {_MC_SAMPLES}")
print("Noise Covariance:\n", np.diag(err_scale).round(3))

Known R assumption: False
Augmented measurement equation: Rate
Augmented coefficient: r_coef
Monte Carlo replications: 100000
Noise Covariance:
 [[3.021 0.    0.   ]
 [0.    4.196 0.   ]
 [0.    0.    0.195]]


In [7]:
print(f"Monte Carlo Ljung-Box summary across {_MC_SAMPLES} replications:")
display(mc_reference["lb_summary"].round(3))

Monte Carlo Ljung-Box summary across 100000 replications:


,measurement,lb_stat,p_value,mc_se_lb_stat,mc_se_p_value,n_replications,n_rejections,reject_rate,reject_rate_mc_se,reject_ci_low,reject_ci_high
0,OutGap,23.332,0.0,0.025,0.0,100000,99949,0.999,0.0,0.999,1.0
1,Infl,36.434,0.0,0.029,0.0,100000,100000,1.000,0.0,1.000,1.0
2,Rate,34.324,0.0,0.029,0.0,100000,100000,1.000,0.0,1.000,1.0


In [8]:
print(f"Moment Tests summary across {_MC_SAMPLES} replications:")
display(mc_reference["moment_specification_test_summary"].round(3))

Moment Tests summary across 100000 replications:


,test,distance,stat,p_value,mc_se_distance,mc_se_stat,mc_se_p_value,n_replications,n_rejections,reject_rate,reject_rate_mc_se,reject_ci_low,reject_ci_high,df,sample_size,bandwidth
0,mean_zero_hac,0.128,1.868,0.663,0.00,0.006,0.001,100000,1568,0.016,0.0,0.015,0.016,3.0,200,4
1,cov_identity,28.202,164.676,0.000,0.01,0.113,0.000,100000,100000,1.000,0.0,1.000,1.000,6.0,200,4


In [9]:
print("Innovations on orthogonalized predicted states (Monte Carlo averages and rejection rates):")
_sort_summary(mc_reference["measurement_regressions_orthogonalized_summary"]).round(3)

Innovations on orthogonalized predicted states (Monte Carlo averages and rejection rates):


,measurement,predictor,coef,standardized_coef,std_error,t_stat,p_value,r2,mc_se_coef,mc_se_standardized_coef,mc_se_std_error,mc_se_t_stat,mc_se_p_value,mc_se_r2,n_replications,n_rejections,reject_rate,reject_rate_mc_se,reject_ci_low,reject_ci_high
0,OutGap,Pi,0.914,0.055,1.186,0.775,0.428,0.007,0.003,0.0,0.000,0.003,0.001,0.0,100000,9604,0.096,0.001,0.094,0.098
1,OutGap,x,-0.724,-0.364,0.131,-5.543,0.000,0.136,0.000,0.0,0.000,0.003,0.000,0.0,100000,99984,1.000,0.000,1.000,1.000
2,OutGap,r,-0.944,-0.036,1.856,-0.505,0.475,0.006,0.006,0.0,0.001,0.003,0.001,0.0,100000,6486,0.065,0.001,0.063,0.066
3,Infl,Pi,-1.544,-0.084,1.282,-1.198,0.320,0.012,0.004,0.0,0.000,0.003,0.001,0.0,100000,21968,0.220,0.001,0.217,0.222
4,Infl,x,0.121,0.056,0.152,0.797,0.410,0.008,0.000,0.0,0.000,0.003,0.001,0.0,100000,12574,0.126,0.001,0.124,0.128
5,Infl,r,-0.839,-0.029,2.014,-0.409,0.472,0.006,0.006,0.0,0.001,0.003,0.001,0.0,100000,7092,0.071,0.001,0.069,0.073
6,Rate,Pi,-0.352,-0.091,0.274,-1.299,0.295,0.013,0.001,0.0,0.000,0.003,0.001,0.0,100000,23985,0.240,0.001,0.237,0.243
7,Rate,x,0.020,0.043,0.033,0.616,0.452,0.006,0.000,0.0,0.000,0.003,0.001,0.0,100000,8290,0.083,0.001,0.081,0.085
8,Rate,r,-1.680,-0.274,0.413,-4.045,0.004,0.079,0.001,0.0,0.000,0.003,0.000,0.0,100000,98285,0.983,0.000,0.982,0.984


In [10]:
print("Innovations on raw predicted states (Monte Carlo averages and rejection rates):")
_sort_summary(mc_reference["measurement_regressions_raw_summary"]).round(3)

Innovations on raw predicted states (Monte Carlo averages and rejection rates):


,measurement,predictor,coef,standardized_coef,std_error,t_stat,p_value,r2,mc_se_coef,mc_se_standardized_coef,mc_se_std_error,mc_se_t_stat,mc_se_p_value,mc_se_r2,n_replications,n_rejections,reject_rate,reject_rate_mc_se,reject_ci_low,reject_ci_high
2,OutGap,Pi,-2.732,-0.206,0.914,-2.980,0.025,0.046,0.003,0.0,0.000,0.003,0.000,0.0,100000,87466,0.875,0.001,0.873,0.877
1,OutGap,x,-0.644,-0.421,0.098,-6.559,0.000,0.179,0.000,0.0,0.000,0.003,0.000,0.0,100000,99999,1.000,0.000,1.000,1.000
0,OutGap,r,1.671,0.067,1.756,0.947,0.394,0.008,0.005,0.0,0.001,0.003,0.001,0.0,100000,10273,0.103,0.001,0.101,0.105
5,Infl,Pi,-0.898,-0.062,1.010,-0.874,0.396,0.009,0.003,0.0,0.000,0.003,0.001,0.0,100000,13777,0.138,0.001,0.136,0.140
4,Infl,x,0.031,0.019,0.117,0.276,0.495,0.005,0.000,0.0,0.000,0.003,0.001,0.0,100000,5366,0.054,0.001,0.052,0.055
3,Infl,r,-1.364,-0.050,1.906,-0.708,0.430,0.007,0.006,0.0,0.001,0.003,0.001,0.0,100000,10348,0.103,0.001,0.102,0.105
8,Rate,Pi,-0.299,-0.099,0.215,-1.410,0.267,0.014,0.001,0.0,0.000,0.003,0.001,0.0,100000,27908,0.279,0.001,0.276,0.282
7,Rate,x,0.017,0.048,0.025,0.685,0.440,0.007,0.000,0.0,0.000,0.003,0.001,0.0,100000,9201,0.092,0.001,0.090,0.094
6,Rate,r,-1.768,-0.305,0.387,-4.548,0.002,0.097,0.001,0.0,0.000,0.003,0.000,0.0,100000,99385,0.994,0.000,0.993,0.994


In [20]:
print("Innovation decomposition orthogonal summary:")
_sort_summary(mc_reference["innovation_decomposition_orthogonalized_summary"])

Innovation decomposition orthogonal summary:


,measurement,predictor,beta_measurement_error,beta_state_prediction_error,beta_total_innovation,beta_component_sum,beta_component_gap,abs_beta_component_gap,reconstruction_max_abs_error,mc_se_beta_measurement_error,mc_se_beta_state_prediction_error,mc_se_beta_total_innovation,mc_se_beta_component_sum,mc_se_beta_component_gap,mc_se_abs_beta_component_gap,mc_se_reconstruction_max_abs_error
0,OutGap,Pi,1.284655,-0.370340,0.914315,0.914315,2.207995e-19,2.840266e-16,1.814973e-15,0.002410,0.001571,0.003416,0.003416,1.185539e-18,7.738116e-19,8.838522e-19
1,OutGap,x,0.051984,-0.776090,-0.724107,-0.724107,-1.564027e-19,8.689674e-17,1.814973e-15,0.000283,0.000224,0.000433,0.000433,3.747760e-19,2.548452e-19,8.838522e-19
2,OutGap,r,-0.429348,-0.514543,-0.943891,-0.943891,-1.072792e-18,4.164859e-16,1.814973e-15,0.003815,0.003025,0.005576,0.005576,1.735069e-18,1.129537e-18,8.838522e-19
3,Infl,Pi,-0.049692,-1.494426,-1.544118,-1.544118,-8.683703e-18,3.442144e-16,1.814973e-15,0.001872,0.003657,0.004086,0.004086,1.460434e-18,9.740509e-19,8.838522e-19
4,Infl,x,0.004061,0.116631,0.120692,0.120692,-8.214759e-19,3.845376e-17,1.814973e-15,0.000222,0.000432,0.000484,0.000484,1.605896e-19,1.049223e-19,8.838522e-19
5,Infl,r,-0.038312,-0.801072,-0.839383,-0.839383,4.745338e-18,4.862528e-16,1.814973e-15,0.002944,0.005840,0.006480,0.006480,2.020713e-18,1.311131e-18,8.838522e-19
6,Rate,Pi,0.002631,-0.354641,-0.352010,-0.352010,-1.277973e-19,1.160079e-16,1.814973e-15,0.000404,0.000726,0.000815,0.000815,4.702395e-19,2.941865e-19,8.838522e-19
7,Rate,x,-0.000413,0.020352,0.019939,0.019939,4.522402e-20,1.317319e-17,1.814973e-15,0.000048,0.000089,0.000099,0.000099,5.301818e-20,3.279638e-20,8.838522e-19
8,Rate,r,-0.019151,-1.661226,-1.680378,-1.680378,1.176871e-18,2.546196e-16,1.814973e-15,0.000630,0.001358,0.001480,0.001480,1.087785e-18,7.314177e-19,8.838522e-19


In [21]:
print("Innovation decomposition raw summary:")
_sort_summary(mc_reference["innovation_decomposition_raw_summary"])

Innovation decomposition raw summary:


,measurement,predictor,beta_measurement_error,beta_state_prediction_error,beta_total_innovation,beta_component_sum,beta_component_gap,abs_beta_component_gap,reconstruction_max_abs_error,mc_se_beta_measurement_error,mc_se_beta_state_prediction_error,mc_se_beta_total_innovation,mc_se_beta_component_sum,mc_se_beta_component_gap,mc_se_abs_beta_component_gap,mc_se_reconstruction_max_abs_error
0,OutGap,Pi,1.561425,-4.293659,-2.732234,-2.732234,4.244001e-19,4.758203e-16,1.814973e-15,0.001919,0.001570,0.002681,0.002681,1.982792e-18,1.291276e-18,8.838522e-19
1,OutGap,x,0.147507,-0.791150,-0.643643,-0.643643,-5.442868e-19,8.455764e-17,1.814973e-15,0.000221,0.000168,0.000330,0.000330,3.629707e-19,2.454589e-19,8.838522e-19
2,OutGap,r,-0.488430,2.158947,1.670517,1.670517,7.312848e-19,4.390788e-16,1.814973e-15,0.003962,0.003436,0.004584,0.004584,1.838254e-18,1.204684e-18,8.838522e-19
3,Infl,Pi,-0.021629,-0.876620,-0.898249,-0.898249,-1.288647e-17,2.579655e-16,1.814973e-15,0.001477,0.002901,0.003220,0.003220,1.080830e-18,7.102003e-19,8.838522e-19
4,Infl,x,0.000924,0.029734,0.030658,0.030658,-1.509458e-18,2.805331e-17,1.814973e-15,0.000171,0.000325,0.000362,0.000362,1.156355e-19,7.432629e-20,8.838522e-19
5,Infl,r,-0.047226,-1.316523,-1.363749,-1.363749,5.085413e-18,4.735701e-16,1.814973e-15,0.002780,0.005431,0.005994,0.005994,1.980509e-18,1.296134e-18,8.838522e-19
6,Rate,Pi,-0.000091,-0.298441,-0.298532,-0.298532,5.378229e-19,9.215628e-17,1.814973e-15,0.000319,0.000578,0.000634,0.000634,3.747754e-19,2.356499e-19,8.838522e-19
7,Rate,x,0.000002,0.017474,0.017476,0.017476,7.389341e-20,1.023508e-17,1.814973e-15,0.000037,0.000071,0.000076,0.000076,4.124142e-20,2.556033e-20,8.838522e-19
8,Rate,r,-0.014164,-1.754007,-1.768171,-1.768171,2.549350e-19,2.541951e-16,1.814973e-15,0.000597,0.001329,0.001446,0.001446,1.089655e-18,7.356568e-19,8.838522e-19


Monte Carlo summaries above aggregate `_MC_SAMPLES` independent draws. The plots and MCMC output below continue on a representative first draw.


In [11]:
parser_aug = ModelParser(_AUGMENTED_CONFIG)
config_aug, kalman_aug = parser_aug.get_all()
solver_aug = DSGESolver(config_aug, kalman_aug)
comp_aug = solver_aug.compile(
    n_state=3,
    n_exog=3,
)
priors = {
    _AUGMENTED_PARAM: make_prior(
        'normal',
        parameters={"mean": 0.0, "std": 4.0, "random_state": next_seed(_plot_seed_counter)},
        transform="identity",
    ),
}

with STDOUT_VOID():
    mc_aug = summarize_mle_augmentation_experiment(
        sol,
        solver_aug,
        comp_aug,
        sol_dgp,
        mc_reference,
        T=T,
        candidate_param=_AUGMENTED_PARAM,
        mc_samples=_MC_SAMPLES,
        alpha=_MC_ALPHA,
        summary_only=_MC_SUMMARY_ONLY,
    )

# estim = lambda: solver_aug.estimate_and_solve(
#     compiled=comp_aug,
#     method="mcmc",
#     n_draws=25_000,
#     burn_in=10_000,
#     thin=2,
#     posterior_point="mean",
#     proposal_scale=1.0,
#     y=obs,
#     priors=priors,
#     steady_state=[0.0, 0.0, 0.0, 0.0, 0.0],
#     random_state=next_seed(_plot_seed_counter),
#     **mc_reference["filter_kwargs"],
# )
# res_aug, sol_aug = estim()

## Diagnostics of the Augmented Model

### Marginal LR Test Conditional on $\theta_0$

In [12]:
print("Monte Carlo LR summary for the MLE-augmented model:")
rep_aug = mc_aug["representative"]
res_mle = rep_aug.res_mle
sol_mle = rep_aug.sol_mle
mle_aug_kf = rep_aug.kf_aug
std_innov_aug_mle = rep_aug.std_innov_aug
sim_aug_mle = rep_aug.sim_aug
mc_aug["lr_summary"].round(3)

Monte Carlo LR summary for the MLE-augmented model:


,estimated_coef,loglik_ref,loglik_aug,lr,p_value,mc_se_estimated_coef,mc_se_loglik_ref,mc_se_loglik_aug,mc_se_lr,mc_se_p_value,n_replications,n_rejections,reject_rate,reject_rate_mc_se,reject_ci_low,reject_ci_high
0,0.639,-4102.203,-4076.77,50.866,0.023,0.001,1.074,1.082,0.123,0.0,100000,94716,0.947,0.001,0.946,0.949


In [13]:
res_mle

OptimizationResult(kind='mle', x=array([0.83201761]), theta={'beta': np.float64(0.971), 'kappa': np.float64(0.58), 'tau_inv': np.float64(1.86), 'psi_pi': np.float64(2.19), 'psi_x': np.float64(0.3), 'rho_r': np.float64(0.84), 'rho_g': np.float64(0.83), 'rho_z': np.float64(0.85), 'pi_star': np.float64(3.43), 'r_star': np.float64(3.01), 'sig_r': np.float64(0.18), 'sig_g': np.float64(0.18), 'sig_z': np.float64(0.64), 'rho_gz': np.float64(0.36), 'meas_infl': np.float64(1e-06), 'meas_rate': np.float64(1e-06), 'meas_outgap': np.float64(1e-06), 'meas_rho_ir': np.float64(0.0), 'meas_rho_gi': np.float64(0.0), 'meas_rho_gr': np.float64(0.0), 'Pi_coef': np.float64(0.0), 'x_coef': np.float64(0.0), 'r_coef': np.float64(0.8320176145674055)}, success=True, message='CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH', fun=np.float64(3565.2536021135675), loglik=np.float64(-3565.2536021135675), logprior=np.float64(0.0), logpost=np.float64(-3565.2536021135675), nfev=12, nit=5, raw=  message: CONVERGENCE

## Serial Autocorrelation Tests for the Augmented Model

In [14]:
print("Monte Carlo Ljung-Box summary for the MLE-augmented model:")
display(mc_aug["lb_summary"].round(3))

Monte Carlo Ljung-Box summary for the MLE-augmented model:


,measurement,lb_stat,p_value,mc_se_lb_stat,mc_se_p_value,n_replications,n_rejections,reject_rate,reject_rate_mc_se,reject_ci_low,reject_ci_high
0,OutGap,23.327,0.0,0.025,0.0,100000,99949,0.999,0.0,0.999,1.0
1,Infl,36.432,0.0,0.029,0.0,100000,100000,1.000,0.0,1.000,1.0
2,Rate,34.084,0.0,0.029,0.0,100000,100000,1.000,0.0,1.000,1.0


In [15]:
print("Reference moment-specification test summary:")
display(mc_reference["moment_specification_test_summary"].round(3))

print("Augmented moment-specification test summary:")
display(mc_aug["moment_specification_test_summary"].round(3))

print("Reference-minus-augmented moment distance comparison:")
display(mc_aug["moment_specification_comparison"].round(3))

Reference moment-specification test summary:


,test,distance,stat,p_value,mc_se_distance,mc_se_stat,mc_se_p_value,n_replications,n_rejections,reject_rate,reject_rate_mc_se,reject_ci_low,reject_ci_high,df,sample_size,bandwidth
0,mean_zero_hac,0.128,1.868,0.663,0.00,0.006,0.001,100000,1568,0.016,0.0,0.015,0.016,3.0,200,4
1,cov_identity,28.202,164.676,0.000,0.01,0.113,0.000,100000,100000,1.000,0.0,1.000,1.000,6.0,200,4


Augmented moment-specification test summary:


,test,distance,stat,p_value,mc_se_distance,mc_se_stat,mc_se_p_value,n_replications,n_rejections,reject_rate,reject_rate_mc_se,reject_ci_low,reject_ci_high,df,sample_size,bandwidth
0,mean_zero_hac,0.124,1.842,0.668,0.00,0.006,0.001,100000,1585,0.016,0.0,0.015,0.017,3.0,200,4
1,cov_identity,28.434,168.189,0.000,0.01,0.119,0.000,100000,100000,1.000,0.0,1.000,1.000,6.0,200,4


Reference-minus-augmented moment distance comparison:


""
